# Logit Attribution Decomposition

Fine-grained decomposition of the final logit vector into per-component, per-position, and per-token contributions.

**References:**
- Elhage et al. (2021) "A Mathematical Framework for Transformer Circuits"
- Nostalgebraist (2020) "Interpreting GPT: the Logit Lens"

## Why This Matters

Full logit decomposition attributes the final logit of any token to specific attention heads, MLP layers, and positions. Cumulative buildup plots show how the prediction assembles layer by layer, revealing which components promote and which suppress each prediction.

**Key references:**
- [Elhage et al. (2021) "A Mathematical Framework for Transformer Circuits"](https://transformer-circuits.pub/2021/framework/index.html) — Foundational framework for attention head and residual stream analysis
- [nostalgebraist (2020) "interpreting GPT: the logit lens"](https://www.lesswrong.com/posts/AcKRB8wDpdaN6v6ru/interpreting-gpt-the-logit-lens) — Project intermediate residuals through the unembedding

In [ ]:
import jax
import jax.numpy as jnp
import numpy as np

from irtk import HookedTransformer, HookedTransformerConfig
from irtk.logit_attribution_decomposition import (
    full_logit_decomposition,
    per_position_logit_attribution,
    top_promoted_demoted_tokens,
    logit_difference_decomposition,
    cumulative_logit_build,
)

cfg = HookedTransformerConfig(
    n_layers=4, d_model=32, n_ctx=64, d_head=8, n_heads=4, d_vocab=100,
)
model = HookedTransformer(cfg)
key = jax.random.PRNGKey(42)
leaves, treedef = jax.tree.flatten(model)
new_leaves = []
for leaf in leaves:
    if isinstance(leaf, jnp.ndarray) and leaf.dtype == jnp.float32:
        key, subkey = jax.random.split(key)
        new_leaves.append(jax.random.normal(subkey, leaf.shape) * 0.1)
    else:
        new_leaves.append(leaf)
model = jax.tree.unflatten(treedef, new_leaves)

tokens = jnp.array([0, 5, 10, 15, 20, 25, 30, 35])
print(f"Model: {cfg.n_layers} layers, {cfg.n_heads} heads")

In [ ]:
# 1. Full logit decomposition
decomp = full_logit_decomposition(model, tokens)
print(f"Target token: {decomp['target_token']}")
print(f"Total logit: {decomp['total_logit']:.4f}")
print(f"Reconstruction error: {decomp['reconstruction_error']:.6f}")
print(f"\nEmbed: {decomp['embed_contribution']:.4f}")
for l in range(cfg.n_layers):
    attn = [f"{decomp['attn_contributions'][l,h]:.3f}" for h in range(cfg.n_heads)]
    print(f"Layer {l}: attn={attn}, mlp={decomp['mlp_contributions'][l]:.3f}")
print(f"Bias: {decomp['bias_contribution']:.4f}")

In [ ]:
# 2. Per-position attribution (which input tokens matter most?)
pos_attr = per_position_logit_attribution(model, tokens)
print(f"Most important position: {pos_attr['most_important_position']}")
for i, (contrib, frac) in enumerate(zip(
    pos_attr['position_contributions'], pos_attr['position_fraction']
)):
    print(f"  Pos {i} (token {int(tokens[i])}): contribution={contrib:.4f}, fraction={frac:.3f}")

In [ ]:
# 3. Top promoted/demoted tokens per component
top = top_promoted_demoted_tokens(model, tokens, top_k=3)
print(f"Embed promotes: {top['embed_promoted']}, demotes: {top['embed_demoted']}")
for l in range(cfg.n_layers):
    for h in range(cfg.n_heads):
        p = top['attn_promoted'][(l,h)]
        d = top['attn_demoted'][(l,h)]
        print(f"  L{l}H{h}: promotes {p}, demotes {d}")

In [ ]:
# 4. Logit difference decomposition
ld = logit_difference_decomposition(model, tokens, token_a=5, token_b=10)
print(f"Logit diff (token 5 - token 10): {ld['logit_diff']:.4f}")
print(f"Largest contributor: {ld['largest_contributor']}")
print(f"Embed diff: {ld['embed_diff']:.4f}")
for l in range(cfg.n_layers):
    attn = [f"{ld['attn_diffs'][l,h]:.3f}" for h in range(cfg.n_heads)]
    print(f"  Layer {l}: attn_diffs={attn}, mlp_diff={ld['mlp_diffs'][l]:.3f}")

In [ ]:
# 5. Cumulative logit build
build = cumulative_logit_build(model, tokens)
print(f"Biggest jump: {build['biggest_jump_component']}")
print(f"Final logit: {build['final_logit']:.4f}")
print(f"\nCumulative build-up:")
for label, cum, delta in zip(
    build['component_labels'], build['cumulative_logits'], build['component_deltas']
):
    print(f"  {label:>8}: delta={delta:+.4f}, cumulative={cum:.4f}")